# The closed loop on the Markevich model

Prompts exactly as in the paper. Each prompt cell starts a fresh session with `new_chat()`; a multi-turn conversation continues in the cells below it without `new_chat()`. Where the paper's run answered a question from Dynalyst, the reply is in a cell after the prompt, as `reply('yes')`: it sends the reply only if Dynalyst has asked a question in this run, and otherwise does nothing.

To install Dynalyst, see `INSTALL.md` at the root of the repository. For what each notebook holds, see `README.md` in this folder.

## How to use
1. Run the **setup cell**, choose the model and paste its API key.
2. Run the **files cell**: it copies the extra models, the R01 time course and the papers from `model_files/` (next to this notebook) into `data/`.
3. Run any prompt cell. Prompts that use BioModels, PubMed or OmniPath need internet.
4. If a cell hangs, use *Kernel > Interrupt*; after a restart, run the setup and files cells again.

In [ ]:
import os, getpass
# Your API key, asked once and kept only in this session (never saved in the notebook).
# Leave it empty if NVIDIA_API_KEY is already set, or to type the key in the box of the setup cell.
if not os.environ.get("NVIDIA_API_KEY"):
    os.environ["NVIDIA_API_KEY"] = getpass.getpass("NVIDIA API key (nvapi-..., or Enter to skip): ").strip()

In [ ]:
import sys
from pathlib import Path

# The project folder: the first folder up from here that holds the dynalyst package.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dynalyst" / "__init__.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dynalyst import config
from dynalyst.env_check import check_env
from dynalyst.model_picker import pick_model
from dynalyst.notebook import ask, show_answer
from dynalyst.agents import unified_agent
from dynalyst.agents.unified_agent import reset_agents as new_chat   # fresh session per prompt


def ask_auto(query):
    """The prompt as the evaluation scripts sent it: no pause before AUTO-07p."""
    answer = unified_agent.ask(query, approval="auto")
    show_answer(answer)
    return answer

def reply(text="yes"):
    """Answer a question Dynalyst asked (which parameter, wait for a long run, a wider range).
    Sends nothing when no question is open, so a rerun that did not ask is not given a stray "yes"."""
    if not any(a.state["pending_approval"] for a in unified_agent._AGENTS.values()):
        print("No open question from Dynalyst in this run: nothing sent.")
        return None
    return ask(text)

check_env()
print("AUTO-07p:", config.AUTO_DIR)
pick_model();     # choose the model, give its API key if asked

In [ ]:
# Copy the extra model files, data and papers into the project (only files not already there)
import shutil
FILES = Path("model_files") if Path("model_files").exists() else ROOT / "notebooks" / "benchmarks" / "model_files"
DEST = {".xml": config.DATA_DIR / "sbml_models", ".ode": config.DATA_DIR / "sbml_models",
        ".csv": config.DATA_DIR / "timecourses", ".pdf": config.PAPERS_DIR}
for f in sorted(FILES.iterdir()):
    d = DEST.get(f.suffix.lower())
    if d and not (d / f.name).exists():
        d.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, d / f.name)
        print("added", d / f.name)
print("model files ready")

## The closed loop on the Markevich model (main text: RQ4, Fig 14; S1 File, Section S5)

One conversation of three turns.

### Closed loop

In [ ]:
new_chat()
ask('In Markevich2004, how does the bistable region in MAPKK change with the total amount of MAPK, and below what total amount does the bistability disappear?')

In [ ]:
reply('yes')   # Dynalyst asked (budget_wait); reply yes

In [ ]:
ask("Now pick a MAPKK value inside the bistable region at the model's own total MAPK, and show how noise makes the system switch between the two states. How does the switching time depend on the system size?")

In [ ]:
reply('yes')   # Dynalyst asked (budget_wait); reply yes

In [ ]:
ask('Finally, reduce Markevich2004 with a quasi-steady-state approximation and check that the reduced model has the same bifurcation diagram in MAPKK as the full model.')